# graph

> Graph-spine construction, commit, and skeptical-lens verification. Builds Document +
> Segment nodes and STARTS_WITH / NEXT / PART_OF edges (pure), commits them through the
> graph-storage capability via the job queue, and re-queries the graph to verify.

In [ ]:
#| default_exp graph

In [ ]:
#| export
from uuid import uuid4
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

from cjm_plugin_system.core.queue import JobQueue, JobStatus
from cjm_graph_plugin_system.core import SourceRef, GraphContext
from cjm_graph_domains.domains.structure import Document, Segment
from cjm_graph_domains.domains.relations import StructureRelations

from cjm_transcript_decomp_core.models import DecompSegment

In [ ]:
#| export
def build_source_ref(
    seg: DecompSegment,  # Aligned segment carrying upstream provenance
    manifest_path: str,  # Path to the consumed transcription run manifest
) -> Optional[SourceRef]:  # Provenance ref, or None when source info is absent
    """Build a manifest-anchored SourceRef for one segment.

    Revolution-1 provenance shape: anchor to the consumed run manifest rather
    than the transcription DB row, because cache-hit transcriptions leave the
    manifest job_id absent from the capability DB (E13 — manifest-vs-row dangle).
    `plugin_name` reuses the `external:<path>` scheme (the canonical SourceRef
    over-fit hack; direct CR-19 generalization evidence). `content_hash` verifies
    the consumed text regardless of whether the row still exists.
    """
    if not seg.source_job_id:
        return None
    slice_str = None
    if seg.start_char is not None and seg.end_char is not None:
        slice_str = f"char:{seg.start_char}-{seg.end_char}"
    return SourceRef(
        plugin_name=f"external:{manifest_path}",
        table_name="transcriptions",
        row_id=seg.source_job_id,
        content_hash=SourceRef.compute_hash(seg.text.encode()),
        segment_slice=slice_str,
    )

In [ ]:
#| export
def build_graph_payload(
    title: str,                     # Document title
    segments: List[DecompSegment],  # Ordered aligned segments (source-coord timing)
    manifest_path: str,             # Consumed manifest path (for SourceRefs)
    media_type: str = "audio",      # Document media type
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], str, List[str]]:  # (nodes, edges, doc_id, seg_ids)
    """Build the (nodes, edges, document_id, segment_ids) graph payload.

    Pure: assembles Document + Segment nodes and STARTS_WITH / NEXT / PART_OF
    edges as wire dicts; no capability calls (commit happens in `commit_graph`).
    """
    doc = Document(id=str(uuid4()), title=title, media_type=media_type)
    doc_node = doc.to_graph_node(sources=[])

    segment_nodes = []
    for seg in segments:
        ref = build_source_ref(seg, manifest_path)
        node = Segment(
            id=str(uuid4()), text=seg.text, index=seg.index,
            start_time=seg.start_time, end_time=seg.end_time,
            start_char=seg.start_char, end_char=seg.end_char,
        ).to_graph_node(sources=[ref] if ref else [])
        segment_nodes.append(node)

    edges: List[Dict[str, Any]] = []
    if segment_nodes:
        edges.append({"id": str(uuid4()), "source_id": doc_node.id,
                      "target_id": segment_nodes[0].id,
                      "relation_type": StructureRelations.STARTS_WITH, "properties": {}})
    for i, node in enumerate(segment_nodes):
        edges.append({"id": str(uuid4()), "source_id": node.id, "target_id": doc_node.id,
                      "relation_type": StructureRelations.PART_OF, "properties": {}})
        if i < len(segment_nodes) - 1:
            edges.append({"id": str(uuid4()), "source_id": node.id,
                          "target_id": segment_nodes[i + 1].id,
                          "relation_type": StructureRelations.NEXT, "properties": {}})

    nodes = [doc_node.to_dict()] + [n.to_dict() for n in segment_nodes]
    return nodes, edges, doc_node.id, [n.id for n in segment_nodes]

In [ ]:
#| export
async def commit_graph(
    queue: JobQueue,              # Started job queue
    graph_id: str,               # Graph-storage capability id
    nodes: List[Dict[str, Any]],  # Node wire dicts from build_graph_payload
    edges: List[Dict[str, Any]],  # Edge wire dicts from build_graph_payload
) -> Dict[str, int]:  # {"nodes": n, "edges": m} created counts
    """Commit nodes then edges to the graph capability via the job queue.

    Goes through `JobQueue` (not a direct `execute_plugin_async`) so graph writes
    get empirical samples + sysmon attribution like every other capability call
    (the GUI GraphService used the direct path — pass-2 two-invocation-paths note).
    """
    async def _run(action, **kw):
        jid = await queue.submit(graph_id, action=action, **kw)
        job = await queue.wait_for_job(jid)
        if job.status != JobStatus.completed:
            raise RuntimeError(f"{graph_id} {action} {job.status}: {job.error}")
        return job.result

    node_res = await _run("add_nodes", nodes=nodes)
    edge_res = await _run("add_edges", edges=edges)
    nc = node_res.get("count", 0) if isinstance(node_res, dict) else 0
    ec = edge_res.get("count", 0) if isinstance(edge_res, dict) else 0
    return {"nodes": nc, "edges": ec}

In [ ]:
#| export
@dataclass
class VerificationResult:
    """Skeptical-lens verification computed by querying the graph directly."""
    document_id: str       # Verified Document node id
    title: str             # Document title (read back from the graph)
    segment_count: int     # Segment nodes found
    has_starts_with: bool  # >=1 STARTS_WITH edge from the Document
    next_chain_complete: bool  # NEXT edge count == segment_count - 1
    part_of_complete: bool     # PART_OF edge count == segment_count
    all_have_timing: bool      # Every segment has start_time + end_time
    all_have_sources: bool     # Every segment has >=1 SourceRef
    source_plugins: List[str] = field(default_factory=list)  # Distinct source plugin_names

    @property
    def ok(self) -> bool:  # True when every structural check passes
        """All structural checks pass."""
        return (self.has_starts_with and self.next_chain_complete
                and self.part_of_complete and self.all_have_timing
                and self.all_have_sources)

In [ ]:
#| export
async def verify_document(
    queue: JobQueue,   # Started job queue
    graph_id: str,     # Graph-storage capability id
    document_id: str,  # Document node id to verify
) -> Optional[VerificationResult]:  # Result, or None if the document is not found
    """Verify a committed document by querying the graph (depth-2 context).

    Skeptical lens: trusts the graph, not the run state — re-reads STARTS_WITH /
    NEXT / PART_OF edges, timing, and source refs from storage. Depth 2 is needed
    so the NEXT edges between discovered Segments are included.
    """
    jid = await queue.submit(graph_id, action="get_context", node_id=document_id, depth=2)
    job = await queue.wait_for_job(jid)
    if job.status != JobStatus.completed:
        raise RuntimeError(f"{graph_id} get_context {job.status}: {job.error}")
    ctx = GraphContext.from_dict(job.result)
    if not ctx.nodes:
        return None

    doc = next((n for n in ctx.nodes if n.label == "Document" and n.id == document_id), None)
    if doc is None:
        return None
    segs = [n for n in ctx.nodes if n.label == "Segment"]
    n = len(segs)

    counts = {"STARTS_WITH": 0, "NEXT": 0, "PART_OF": 0}
    for e in ctx.edges:
        if e.relation_type in counts:
            counts[e.relation_type] += 1

    missing_timing = sum(1 for s in segs
                         if s.properties.get("start_time") is None
                         or s.properties.get("end_time") is None)
    plugins = set()
    missing_sources = 0
    for s in segs:
        if not s.sources:
            missing_sources += 1
        for src in s.sources:
            name = (src.get("plugin_name") if isinstance(src, dict)
                    else getattr(src, "plugin_name", None))
            if name:
                plugins.add(name)

    return VerificationResult(
        document_id=document_id,
        title=doc.properties.get("title", "Untitled"),
        segment_count=n,
        has_starts_with=counts["STARTS_WITH"] >= 1,
        next_chain_complete=counts["NEXT"] == max(0, n - 1),
        part_of_complete=counts["PART_OF"] == n,
        all_have_timing=missing_timing == 0,
        all_have_sources=missing_sources == 0,
        source_plugins=sorted(plugins),
    )

In [ ]:
# graph payload smoke checks (pure; no capabilities)
_segs = [
    DecompSegment(index=0, text="Alpha.", start_time=0.0, end_time=2.0,
                  start_char=0, end_char=6, source_job_id="j0", source_provider_id="p"),
    DecompSegment(index=1, text="Beta.", start_time=2.0, end_time=4.0,
                  start_char=0, end_char=5, source_job_id="j1", source_provider_id="p"),
    DecompSegment(index=2, text="Gamma.", start_time=4.0, end_time=6.0,
                  start_char=0, end_char=6, source_job_id="j2", source_provider_id="p"),
]
nodes, edges, doc_id, seg_ids = build_graph_payload("Test Doc", _segs, "/tmp/m.json")
assert len(nodes) == 4  # 1 Document + 3 Segments
assert len(seg_ids) == 3
_rels = [e["relation_type"] for e in edges]
assert _rels.count("STARTS_WITH") == 1
assert _rels.count("NEXT") == 2
assert _rels.count("PART_OF") == 3

_sr = build_source_ref(_segs[0], "/tmp/m.json")
assert _sr.plugin_name == "external:/tmp/m.json"
assert _sr.row_id == "j0"
assert _sr.segment_slice == "char:0-6"
assert build_source_ref(DecompSegment(0, "x", 0.0, 1.0, None, None, None, None), "/tmp/m.json") is None
print("graph payload checks OK")

graph payload checks OK
